# Natura forest reference data — local plots and spatial sampling

**Purpose:** prepare independent 20 m monitoring plots, optional weakly labelled points inside Natura inventory polygons, and geographically grouped train/validation/test assignments. The output is ready for a later Sentinel-1/2 and LiDAR neighbourhood-feature workflow.

**Scientific design:** A monitoring assessment belongs to the 20 m circular plot. Polygon-derived points inherit *polygon-level* labels and are **not independently verified local observations**. Their records are kept separate from monitoring observations. Do not include `plot_id`, `polygon_id`, `block_id`, or `fold_id` as model predictors.

**What this notebook does not do:** download rasters, calculate focal/neighbourhood features, train models, or establish accuracy outside the inventoried/monitored population.

1. Change the input paths and settings below.
2. Run all cells (Python environment needs `pandas`, `numpy`, `geopandas`, `shapely`, `scipy`, `scikit-learn`). No `osgeo` import is required.
3. Inspect the GeoPackages and the CSV summary in QGIS. Only then begin raster extraction.

Geographic blocks and the 1 km test-buffer distance are **initial design choices**, not demonstrated autocorrelation ranges. Revise them later based on spatial coverage and predictor/residual autocorrelation. Splits must be fixed before inspecting any model scores.

In [ ]:
from pathlib import Path

# --- Paths: edit these on your computer ---
MONITORING_CSV = Path(r"detailsed_seireandmed.csv")
INVENTORY_FILE = Path(r"C:\Users\kaursa40847\project\working\natura_habitatsPolygon_forests.gpkg")
INVENTORY_LAYER = "natura_habitatsPolygon_forests"  # Set None for the first/default layer.
OUTPUT_DIR = Path.cwd() / "natura_reference_output"

# --- Data and geographic design ---
YEAR_WINDOW = (2017, 2026)
SPATIAL_BLOCK_M = 10_000         # Initial value; NOT an estimated autocorrelation range.
N_FOLDS = 5                     # Fold 0 = final test; folds 1–4 = train/CV.
TEST_FOLD = 0
BALANCE_FIELD = "structural_preservation"  # Can also use "habitat_type" or "overall_conservation_value".
TEST_BUFFER_M = 1_000           # Removes candidate training samples near held-out test samples.
SEED = 42

# --- Optional inventory-derived local samples (weak labels) ---
MAKE_POLYGON_SAMPLES = True
MAX_SAMPLES_PER_POLYGON = 6
INNER_BUFFER_M = 20             # Try to avoid mixed boundary pixels.
MIN_POINT_DISTANCE_M = 60       # Minimum separation WITHIN each polygon.

# Explicit output files in OUTPUT_DIR are regenerated on each run; input files are never changed.

In [ ]:
import hashlib
import math
import re
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from shapely.geometry import MultiPolygon, Point, Polygon, box
from shapely.validation import make_valid
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.max_columns", 30)
CRS = "EPSG:3301"
STARRED_TYPES = {"9010", "9020", "9080", "9180", "91D0", "91E0"}

PLOT_COLUMNS = {
    "Elupaigatüüp (LD)": "habitat_type",
    "Elupaiga esinduslikkus": "representativeness",
    "Koosluse struktuuri säilimine": "structural_preservation",
    "Elupaiga funktsioonide säilimine": "functional_preservation",
    "Elupaiga üldine looduskaitseline väärtus": "overall_conservation_value",
    "Kasvukohatüüp (Paal 1997)": "growth_site_type",
}

INVENTORY_FIELDS = {
    "polygon_id": "id",
    "habitat_type": "pohityyp",
    "representativeness": "esinduslik",
    "structural_preservation": "str_sail",
    "functional_preservation": "funkt_sail",
    "restorability": "taastatavs",
    "conservation_status": "lk_seisund",  # No direct equivalent in the monitoring CSV.
    "overall_conservation_value": "uldine_lk",
    "inventory_date": "invent_kpv",
    "protected_area": "kaitst_ala",
}


def habitat_code(value):
    if pd.isna(value):
        return None
    match = re.match(r"^(?:Eesti\s+)?(0|\d[0-9A-Z]{3}\*?)", str(value).strip(), re.I)
    if not match:
        return None
    value = match.group(1).upper().rstrip("*")
    return value + ("*" if value in STARRED_TYPES else "")


def rating_code(value):
    if pd.isna(value):
        return None
    match = re.match(r"^\s*(IV|III|II|I|[A-DP])(?=\s|[-.:]|$)", str(value).upper())
    return match.group(1) if match else None


def spatial_block(point):
    return f"B{math.floor(point.x / SPATIAL_BLOCK_M)}_{math.floor(point.y / SPATIAL_BLOCK_M)}"


def stable_seed(key):
    token = hashlib.sha256(f"{SEED}:{key}".encode("utf-8")).hexdigest()
    return int(token[:16], 16)


def save_gpkg(frame, filename, layer):
    if len(frame) == 0:
        print(f"Skipped empty layer: {filename}")
        return
    frame.to_file(OUTPUT_DIR / filename, layer=layer, driver="GPKG", index=False)


def source_dates(series):
    return pd.to_datetime(series, errors="coerce", dayfirst=True, format="mixed")

## 1. Convert the KESE monitoring CSV into one row per circular plot

`detailsed_seireandmed.csv` is in **long format** (several indicator rows for one observation). The observed habitat is taken from `Elupaigatüüp (LD)`, **not** the programme's `Seirekogumi nimi`. The exported data distinguish measured labels, missing labels and other responses such as `0 elupaigatüüp`.

**Coordinate caution:** in this extract, `Sisestatud x L-EST97` contains northing (~6.5 million) and `Sisestatud y L-EST97` contains easting (~600,000), opposite the usual GIS x/y interpretation. The function below swaps them and validates the result.

In [ ]:
def prepare_monitoring():
    if not MONITORING_CSV.exists():
        raise FileNotFoundError(f"Monitoring file not found: {MONITORING_CSV}")

    raw = pd.read_csv(MONITORING_CSV, sep=";", dtype=str, encoding="utf-8-sig")
    forest = raw[raw["Seiretöö nimetus"].str.startswith("Metsaelupaikade seire", na=False)].copy()
    obs_id = "Proovi / vaatluse kood"
    variable = "Näitaja nimetus"
    value = "Väärtus (muud)"
    forest = forest.dropna(subset=[obs_id])

    # Check that the observations truly use the 20 m field protocol.
    radius = forest["Vaatlusgrupp"].str.extract(r"r\s*=\s*(\d+)m", expand=False)
    other = radius.ne("20")
    if other.any():
        warnings.warn(f"Skipping {int(other.sum())} records without an explicit r=20m protocol.")
        forest = forest.loc[~other].copy()

    disagreements = forest.groupby([obs_id, variable])[value].nunique(dropna=True)
    if (disagreements > 1).any():
        warnings.warn("Some observations contain conflicting values for the same indicator; see output QC file.")
    conflicts = disagreements[disagreements > 1].rename("n_distinct_values").reset_index()

    keep = [obs_id, "Seirekoha nimi", "Seirekogumi nimi", "Seireaja algus",
            "Sisestatud x L-EST97", "Sisestatud y L-EST97"]
    details = forest.groupby(obs_id, as_index=False)[keep[1:]].first()
    values = forest.pivot_table(index=obs_id, columns=variable, values=value, aggfunc="first").reset_index()
    wide = details.merge(values, on=obs_id, how="left", validate="one_to_one")

    plots = pd.DataFrame({
        "plot_id": wide[obs_id].astype(str),
        "location_name": wide["Seirekoha nimi"],
        "monitoring_group": wide["Seirekogumi nimi"],
        "inventory_date": wide["Seireaja algus"],
        # The CSV's coordinate names are reversed; see the markdown above.
        "easting": pd.to_numeric(wide["Sisestatud y L-EST97"].str.replace(",", ".", regex=False), errors="coerce"),
        "northing": pd.to_numeric(wide["Sisestatud x L-EST97"].str.replace(",", ".", regex=False), errors="coerce"),
    })
    for original, standard in PLOT_COLUMNS.items():
        plots[f"raw_{standard}"] = wide[original] if original in wide else None
        plots[standard] = (plots[f"raw_{standard}"].map(habitat_code)
                           if standard == "habitat_type" else
                           plots[f"raw_{standard}"] if standard == "growth_site_type" else
                           plots[f"raw_{standard}"].map(rating_code))

    plots["inventory_year"] = source_dates(plots["inventory_date"]).dt.year
    plots["in_year_window"] = plots["inventory_year"].between(*YEAR_WINDOW).fillna(False)
    plots["plot_radius_m"] = 20
    plots["reference_source"] = "field_plot"
    plots["label_scope"] = "20m_plot"

    valid_xy = plots["easting"].between(300_000, 800_000) & plots["northing"].between(6_300_000, 6_700_000)
    rejected = plots.loc[~valid_xy].copy()
    plots = plots.loc[valid_xy].reset_index(drop=True)
    centres = gpd.GeoDataFrame(plots, geometry=gpd.points_from_xy(plots.easting, plots.northing), crs=CRS)
    centres["block_id"] = centres.geometry.map(spatial_block)
    print(f"Forest monitoring observations: {len(centres):,}; rejected coordinates: {len(rejected):,}")
    print("Measured target availability:")
    print(centres[["habitat_type", "structural_preservation", "overall_conservation_value"]].notna().sum())
    return centres, rejected, conflicts

## 2. Optionally read the polygon inventory without changing it

The original source geometries and attributes remain untouched. We retain the original assessments alongside standardized names. Polygons have heterogeneous condition; points derived from them later are **weakly labelled**, not new field observations.

In [ ]:
def polygons_only(geometry):
    if geometry is None or geometry.is_empty:
        return None
    if not geometry.is_valid:
        geometry = make_valid(geometry)
    if isinstance(geometry, (Polygon, MultiPolygon)):
        return geometry if not geometry.is_empty else None
    if hasattr(geometry, "geoms"):
        parts = [g for g in geometry.geoms if isinstance(g, Polygon) and not g.is_empty]
        return MultiPolygon(parts) if parts else None
    return None


def prepare_inventory():
    if not INVENTORY_FILE.exists():
        warnings.warn(f"Inventory file unavailable. Plot-only workflow will run: {INVENTORY_FILE}")
        return None

    options = {"layer": INVENTORY_LAYER} if INVENTORY_LAYER else {}
    raw = gpd.read_file(INVENTORY_FILE, **options)
    if raw.crs is None:
        raise ValueError("Inventory has no CRS. Define its correct source CRS before continuing.")
    raw = raw.to_crs(CRS).reset_index(drop=True)

    result = pd.DataFrame(index=raw.index)
    for standard, original in INVENTORY_FIELDS.items():
        result[standard] = raw[original] if original in raw else None
        if original not in raw:
            warnings.warn(f"Inventory field unavailable: {original} ({standard})")
    result["polygon_row_id"] = np.arange(len(raw), dtype=int)
    result["polygon_id"] = result["polygon_id"].astype("string")
    missing_ids = result["polygon_id"].isna()
    result.loc[missing_ids, "polygon_id"] = [f"source_row_{i}" for i in result.index[missing_ids]]

    for target in ["representativeness", "structural_preservation", "functional_preservation",
                   "conservation_status", "overall_conservation_value"]:
        result[f"raw_{target}"] = result[target]
        result[target] = result[target].map(rating_code)
    result["raw_habitat_type"] = result["habitat_type"]
    result["habitat_type"] = result["habitat_type"].map(habitat_code)
    result["inventory_year"] = source_dates(result["inventory_date"]).dt.year
    result["in_year_window"] = result["inventory_year"].between(*YEAR_WINDOW).fillna(False)
    result["reference_source"] = "inventory_polygon"
    result["label_scope"] = "whole_polygon"

    out = gpd.GeoDataFrame(result, geometry=raw.geometry.map(polygons_only), crs=CRS)
    unusable = out.geometry.isna() | out.geometry.is_empty
    if unusable.any():
        warnings.warn(f"Excluding {int(unusable.sum())} unusable polygon geometries.")
    out = out.loc[~unusable].copy().reset_index(drop=True)
    out["area_ha"] = out.geometry.area / 10_000
    out = out[out.area_ha > 0].copy().reset_index(drop=True)
    out["block_id"] = out.geometry.representative_point().map(spatial_block)
    print(f"Inventory polygons: {len(out):,}; inventory area: {out.area_ha.sum():,.1f} ha")
    return out

## 3. Make geographic folds before sampling polygon interiors

`fold_id = 0` is the final held-out test. Folds 1–4 are for spatial cross-validation during model selection. Blocks are assigned using **field plots** (the primary evaluation source); inventory-only blocks get deterministic fold assignments. Target labels can be changed in the settings, but **do not change the splits after inspecting model results**.

When a monitoring centre falls within an inventory polygon, the polygon inherits the plot's fold. If a polygon covers plots from different folds, that polygon is excluded from optional weak-label sampling, and the relevant field plots are flagged to prevent direct cross-fold leakage. Boundary proximity is checked again after sampling.

In [ ]:
def assign_geographic_folds(centres, inventory):
    if centres.block_id.nunique() < N_FOLDS:
        raise ValueError(f"Only {centres.block_id.nunique()} spatial groups; cannot make {N_FOLDS} folds.")
    classes = centres[BALANCE_FIELD].fillna("UNKNOWN").astype(str)
    splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_by_block = {}
    for fold, (_, validation_idx) in enumerate(splitter.split(centres, classes, groups=centres.block_id)):
        for block in centres.iloc[validation_idx].block_id.unique():
            fold_by_block[block] = fold

    centres = centres.copy()
    centres["fold_id"] = centres.block_id.map(fold_by_block).astype(int)
    centres["split"] = np.where(centres.fold_id == TEST_FOLD, "test", "train_cv")
    if inventory is None:
        return centres, None, pd.DataFrame()

    inventory = inventory.copy()
    extra = inventory.loc[~inventory.block_id.isin(fold_by_block), "block_id"].unique()
    for block in sorted(extra):
        fold_by_block[block] = stable_seed(block) % N_FOLDS
    inventory["fold_id"] = inventory.block_id.map(fold_by_block).astype(int)
    inventory["split"] = np.where(inventory.fold_id == TEST_FOLD, "test", "train_cv")

    # Monitoring plots inside inventory polygons must never leak across folds.
    checked_fields = ["habitat_type", "structural_preservation", "overall_conservation_value", "inventory_year"]
    plot_columns = centres[["plot_id", "block_id", "fold_id", *checked_fields, "geometry"]]
    polygon_columns = inventory[["polygon_row_id", "polygon_id", *checked_fields, "geometry"]]
    matches = gpd.sjoin(plot_columns, polygon_columns, how="inner", predicate="intersects")
    links = matches.drop(columns="geometry", errors="ignore").rename(columns={
        "habitat_type_left": "plot_habitat_type",
        "habitat_type_right": "inventory_habitat_type",
    })
    for field in ["habitat_type", "structural_preservation", "overall_conservation_value"]:
        left = "plot_habitat_type" if field == "habitat_type" else f"{field}_left"
        right = "inventory_habitat_type" if field == "habitat_type" else f"{field}_right"
        links[f"{field}_agrees"] = np.where(
            links[left].notna() & links[right].notna(), links[left].eq(links[right]), pd.NA
        )
    links["inventory_year_difference"] = (
        pd.to_numeric(links["inventory_year_left"], errors="coerce")
        - pd.to_numeric(links["inventory_year_right"], errors="coerce")
    )
    conflicting_polygon_ids = set()
    conflicting_plot_ids = set()

    for polygon_row_id, matched in matches.groupby("polygon_row_id"):
        fold_values = matched.fold_id.unique()
        block_values = matched.block_id.unique()
        if len(fold_values) == 1 and len(block_values) == 1:
            mask = inventory.polygon_row_id.eq(polygon_row_id)
            inventory.loc[mask, "fold_id"] = int(fold_values[0])
            inventory.loc[mask, "block_id"] = block_values[0]
            inventory.loc[mask, "split"] = "test" if fold_values[0] == TEST_FOLD else "train_cv"
        else:
            conflicting_polygon_ids.add(polygon_row_id)
            conflicting_plot_ids.update(matched.plot_id)

    inventory.loc[inventory.polygon_row_id.isin(conflicting_polygon_ids), "split"] = "overlap_excluded"
    centres.loc[centres.plot_id.isin(conflicting_plot_ids), "split"] = "overlap_excluded"

    if conflicting_polygon_ids:
        warnings.warn(f"Excluded {len(conflicting_polygon_ids)} cross-fold polygons and "
                      f"{len(conflicting_plot_ids)} affected plots from modelling.")
    print(f"Plot-to-inventory matches: {len(links):,}; unique matched plots: {matches.plot_id.nunique():,}")
    return centres, inventory, links


def exclude_near_test(centres, samples=None, buffer_m=TEST_BUFFER_M):
    """Mark, rather than delete, candidate training observations close to final test samples."""
    frames = [centres] + ([samples] if samples is not None and len(samples) else [])
    test_points = [frame.loc[frame.split == "test", "geometry"] for frame in frames]
    test_geometries = [geom for series in test_points for geom in series]
    if len(test_geometries) == 0:
        raise ValueError("No held-out test observations remain after QC.")
    test_xy = np.asarray([(geom.x, geom.y) for geom in test_geometries])
    tree = cKDTree(test_xy)
    updated = []
    for frame in frames:
        frame = frame.copy()
        is_train = frame.split.eq("train_cv")
        if is_train.any():
            candidate = frame.loc[is_train, "geometry"]
            xy = np.asarray([(geom.x, geom.y) for geom in candidate])
            too_close = tree.query(xy, k=1)[0] < buffer_m
            frame.loc[candidate.index[too_close], "split"] = "buffer_excluded"
        updated.append(frame)
    return updated[0], (updated[1] if len(updated) > 1 else None)


def buffered_cv_indices(samples, validation_fold, buffer_m=TEST_BUFFER_M):
    """Use after raster extraction; removes training rows near validation rows for that CV fold."""
    candidate = samples.loc[(samples.split == "train_cv") & (samples.fold_id != TEST_FOLD)]
    validation = candidate.loc[candidate.fold_id == validation_fold]
    training = candidate.loc[candidate.fold_id != validation_fold]
    if validation.empty or training.empty:
        raise ValueError("Empty fold: check class and geographical distribution.")
    if buffer_m:
        val_xy = np.asarray([(geom.x, geom.y) for geom in validation.geometry])
        train_xy = np.asarray([(geom.x, geom.y) for geom in training.geometry])
        keep = cKDTree(val_xy).query(train_xy, k=1)[0] >= buffer_m
        training = training.iloc[np.flatnonzero(keep)]
    return training.index.to_numpy(), validation.index.to_numpy()

## 4. Optional spatially distributed points inside inventory polygons

Each polygon contributes a small capped number of separated points; a point inherits the polygon's assessment. These are explicitly tagged `inventory_weak`. Points are *not* local ground truth and should never form the primary accuracy test. The total statistical weight of each polygon is approximately 1, regardless of its number of sampled points.

This step is optional and independent of the monitoring-plot baseline. The number of points is an adjustable computational compromise, not a formally optimized survey design.

In [ ]:
def points_in_polygon(geometry, count, seed_key):
    inner = geometry.buffer(-INNER_BUFFER_M)
    if inner.is_empty or inner.area <= 0:
        return [geometry.representative_point()], True
    rng = np.random.default_rng(stable_seed(seed_key))
    xmin, ymin, xmax, ymax = inner.bounds
    selected = [inner.representative_point()]
    min_distance_sq = MIN_POINT_DISTANCE_M ** 2

    # Random candidates are accepted only when interior and sufficiently separated.
    for _ in range(200):
        if len(selected) >= count:
            break
        point = Point(rng.uniform(xmin, xmax), rng.uniform(ymin, ymax))
        if not inner.contains(point):
            continue
        if all((point.x - old.x) ** 2 + (point.y - old.y) ** 2 >= min_distance_sq
               for old in selected):
            selected.append(point)
    return selected, False


def sample_inventory(inventory):
    if inventory is None or not MAKE_POLYGON_SAMPLES:
        return None
    fields = ["polygon_row_id", "polygon_id", "habitat_type", "representativeness",
              "structural_preservation", "functional_preservation", "restorability",
              "conservation_status", "overall_conservation_value", "inventory_year",
              "inventory_date", "area_ha", "block_id", "fold_id", "split", "protected_area"]
    rows = []
    for record in inventory.itertuples(index=False):
        if record.split == "overlap_excluded":
            continue
        count = min(MAX_SAMPLES_PER_POLYGON, max(1, math.ceil(1.5 * math.sqrt(record.area_ha))))
        points, edge_fallback = points_in_polygon(record.geometry, count, record.polygon_row_id)
        values = {field: getattr(record, field) for field in fields}
        values.update(reference_source="inventory_weak", label_scope="whole_polygon_inherited",
                      edge_fallback=edge_fallback, n_points_in_polygon=len(points),
                      sample_weight=1 / len(points))
        for number, point in enumerate(points, 1):
            rows.append({**values, "sample_id": f"inv_{record.polygon_row_id}_{number}", "geometry": point})

    if not rows:
        print("No eligible polygons for optional sampling.")
        return None
    samples = gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS)
    print(f"Optional inventory-derived points: {len(samples):,} from "
          f"{samples.polygon_row_id.nunique():,} polygons")
    return samples

## 5. Export spatial references and check class/fold coverage

Exports use explicit new filenames inside `OUTPUT_DIR` and never modify the supplied CSV or source inventory. Re-running this notebook regenerates **only** the named output files within that directory.

Review `reference_split_summary.csv`, `plot_class_by_fold.csv`, the monitoring map, and particularly the number of independent blocks per habitat and target class. A blocked fold cannot reliably evaluate a class absent from its test samples.

In [ ]:
def export_results(centres, inventory, samples, links, rejected, conflicts):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    save_gpkg(centres, "monitoring_centres.gpkg", "monitoring_centres")
    circles = centres.copy()
    circles.geometry = circles.geometry.buffer(20, resolution=24)
    save_gpkg(circles, "monitoring_plots_20m.gpkg", "monitoring_plots_20m")
    if inventory is not None:
        save_gpkg(inventory, "inventory_polygons_split.gpkg", "inventory_polygons")
    if samples is not None and len(samples):
        save_gpkg(samples, "inventory_sample_points.gpkg", "inventory_samples")

    if len(links):
        links.to_csv(OUTPUT_DIR / "monitoring_inventory_links.csv", index=False)
    rejected.to_csv(OUTPUT_DIR / "monitoring_invalid_coordinates.csv", index=False)
    conflicts.to_csv(OUTPUT_DIR / "monitoring_conflicting_indicators.csv", index=False)

    groups = pd.concat([centres[["block_id", "fold_id"]],
                        inventory[["block_id", "fold_id"]] if inventory is not None else
                        centres[["block_id", "fold_id"]].iloc[:0]]).drop_duplicates()
    # A group can appear with different fold IDs only where a polygon was linked to a plot.
    group_duplicates = groups.block_id.duplicated(keep=False)
    if group_duplicates.any():
        warnings.warn("Some geographic blocks have linked polygons assigned to neighbouring folds; "
                      "use the per-record fold and the exclusion flags rather than only block names.")
    rows = []
    for row in groups.itertuples(index=False):
        col, north = map(int, row.block_id.removeprefix("B").split("_"))
        x, y = col * SPATIAL_BLOCK_M, north * SPATIAL_BLOCK_M
        rows.append({"block_id": row.block_id, "fold_id": row.fold_id,
                     "geometry": box(x, y, x + SPATIAL_BLOCK_M, y + SPATIAL_BLOCK_M)})
    group_geo = gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS)
    save_gpkg(group_geo, "spatial_groups.gpkg", "spatial_groups")

    sources = [centres] + ([samples] if samples is not None else [])
    overall = pd.concat([x.drop(columns="geometry") for x in sources], ignore_index=True)
    summary = overall.groupby(["reference_source", "split", "fold_id"], dropna=False).agg(
        n_records=("reference_source", "size"),
        n_known_habitat=("habitat_type", "count"),
        n_known_structure=("structural_preservation", "count"),
        n_known_overall_value=("overall_conservation_value", "count"),
        n_blocks=("block_id", "nunique"),
    ).reset_index()
    summary.to_csv(OUTPUT_DIR / "reference_split_summary.csv", index=False)
    for target in ["habitat_type", "structural_preservation", "overall_conservation_value"]:
        counts = pd.crosstab(centres[target].fillna("UNKNOWN"), centres["fold_id"])
        counts.to_csv(OUTPUT_DIR / f"plot_{target}_by_fold.csv")
    pd.crosstab(centres.inventory_year.fillna(-1), centres.split).to_csv(
        OUTPUT_DIR / "monitoring_year_by_split.csv"
    )
    if inventory is not None:
        inventory.groupby("habitat_type", dropna=False).agg(
            n_polygons=("polygon_row_id", "size"), area_ha=("area_ha", "sum")
        ).to_csv(OUTPUT_DIR / "inventory_habitat_distribution.csv")
    print("\nPrimary reference distribution by split:")
    print(centres.groupby(["split", "fold_id"]).size().to_string())
    print(f"\nFinished. Files are in: {OUTPUT_DIR}")
    return summary

## 6. Run data preparation

Start with the monitoring CSV. If the inventory path does not exist on this computer, the monitoring-only outputs still run and the inventory-dependent steps are skipped. Check the printed class distribution and study-region coverage before accepting the split design.

In [ ]:
monitoring_centres, rejected_coordinates, indicator_conflicts = prepare_monitoring()
inventory_polygons = prepare_inventory()
monitoring_centres, inventory_polygons, plot_polygon_links = assign_geographic_folds(
    monitoring_centres, inventory_polygons
)
inventory_samples = sample_inventory(inventory_polygons)
monitoring_centres, inventory_samples = exclude_near_test(
    monitoring_centres, inventory_samples, buffer_m=TEST_BUFFER_M
)
reference_summary = export_results(
    monitoring_centres, inventory_polygons, inventory_samples,
    plot_polygon_links, rejected_coordinates, indicator_conflicts
)
display(reference_summary)

## 7. How to use these outputs for remote-sensing modelling

1. For the **primary model**, use monitoring centres and the corresponding measured `structural_preservation`, `habitat_type`, or `overall_conservation_value`. The CSV does **not** have a direct `conservation_status` (`lk_seisund`) field, so do not invent one.
2. Extract **local** raster features from the 20 m reference circle. Compute **neighbourhood** statistics using moving windows centred at the same point (e.g. 50 m and 150 m); no extra circles need to be stored.
3. Use training records with `split == 'train_cv'` for fitting; hold back `split == 'test'` until final evaluation. Rows marked `buffer_excluded` or `overlap_excluded` are retained only for QC.
4. For cross-validation, validate one `fold_id` among 1–4 at a time and remove nearby training records using `buffered_cv_indices`. All records from a polygon must stay in its assigned fold.
5. The `inventory_weak` points are **optional extra training data**. Train and evaluate the monitoring-only baseline first. Only combine sources if the target assessment definitions and survey dates are compatible.
6. Validate **local** predictions using held-out monitoring plots. A supplementary polygon-level evaluation can aggregate independent predictions, but it is not equivalent to validating each local prediction.
7. Compare training predictors with predictors at prediction locations before claiming transferability to unmonitored forests; spatial CV does not establish accuracy outside the sampled population.

**Important:** 10 km blocks and 1 km buffers are starting choices. Evaluate fold maps, residual spatial dependence and rare-class coverage; do not interpret these values as guarantees of spatial independence. The 20 m plot footprint is an inferred interpretation of the field method `Botaaniline seire (r=20m)` and should be checked against the official monitoring protocol.